In [2]:
import sys
print(sys.version)

3.12.13 (main, Jul 18 2026, 17:02:19) [Clang 22.1.3 ]


In [1]:
import requests

url = "https://api.etherscan.io/v2/api?module=proxy&action=eth_getTransactionByHash&apikey=Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

response = requests.get(url)

print(response.text)

{"status":"0","message":"NOTOK","result":"Missing chainid parameter (required for v2 api), please see https://api.etherscan.io/v2/chainlist for the list of supported chainids"}


In [2]:
import requests

url = "https://api.etherscan.io/v2/chainlist"

response = requests.get(url)

print(response.text)

{
  "comments": "List of API endpoints maintained by Etherscan EAAS. Available Status codes are (0)=Offline, (1)=Ok, (2)=Degraded",
  "totalcount": 61,
  "result": [
    {
      "chainname": "Ethereum Mainnet",
      "chainid": "1",
      "blockexplorer": "https://etherscan.io/",
      "apiurl": "https://api.etherscan.io/v2/api?chainid=1",
      "status": 1,
      "comment": ""
    },
    {
      "chainname": "Sepolia Testnet",
      "chainid": "11155111",
      "blockexplorer": "https://sepolia.etherscan.io/",
      "apiurl": "https://api.etherscan.io/v2/api?chainid=11155111",
      "status": 1,
      "comment": ""
    },
    {
      "chainname": "Hoodi Testnet",
      "chainid": "560048",
      "blockexplorer": "https://hoodi.etherscan.io/",
      "apiurl": "https://api.etherscan.io/v2/api?chainid=560048",
      "status": 1,
      "comment": ""
    },
    {
      "chainname": "BNB Smart Chain Mainnet",
      "chainid": "56",
      "blockexplorer": "https://bscscan.com/",
      "apiur

In [3]:
import requests
import csv

url = "https://api.etherscan.io/v2/chainlist"

response = requests.get(url)
data = response.json()

chains = data["result"]

output_file = "../data/chainlist.csv"

with open(output_file, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            "chainname",
            "chainid",
            "blockexplorer",
            "apiurl",
            "status",
            "comment"
        ]
    )

    writer.writeheader()
    writer.writerows(chains)

print(f"Saved {len(chains)} chains to {output_file}")

Saved 61 chains to ../data/chainlist.csv


In [4]:
### web site ka data aa raha hai pura

import requests

url = "https://etherscan.io/"

response = requests.get(url)

print(response.text)

<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-mkO273PmDMq86fa1kNwDLD&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-src-attr &#39;none&#39;; style-src &#39;unsafe-inline&#39;; img-src &#39;self&#39; https://challenges.cloudflare.com; connect-src &#39;self&#39; https://challenges.cloudflare.com; frame-src &#39;self&#39; https://challenges.cloudflare.com blob:; child-src &#39;self&#39; https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri &#39;self&#39;"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font

In [5]:
import requests
import csv
import os

API_KEY = "Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

BLOCK_NUMBER = 11645249

url = "https://api.etherscan.io/v2/api"

params = {
    "chainid": "11155111",       # Sepolia
    "module": "proxy",
    "action": "eth_getBlockByNumber",
    "tag": hex(BLOCK_NUMBER),    # 11645249 -> 0xb1a601
    "boolean": "true",
    "apikey": API_KEY,
}

response = requests.get(url, params=params)
data = response.json()

block = data["result"]
transactions = block["transactions"]

print(f"Block: {BLOCK_NUMBER}")
print(f"Transactions found: {len(transactions)}")

# Create ../data if it doesn't exist
os.makedirs("../data", exist_ok=True)

output_file = "../data/sepolia_transactions.csv"

with open(output_file, "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    writer.writerow([
        "tx_hash",
        "from",
        "to",
        "value",
        "block_number"
    ])

    for tx in transactions:

        writer.writerow([
            tx["hash"],
            tx["from"],
            tx["to"],
            tx["value"],
            BLOCK_NUMBER
        ])

print(f"Saved to {output_file}")

Block: 11645249
Transactions found: 112
Saved to ../data/sepolia_transactions.csv


In [1]:
# for appending more data bcz uppper one only look in one block

import requests
import csv
import os
import time

API_KEY = "Z7JQRN2EP5SITUXU7CTPAZEHZ51U19E4J2"

# Sepolia
CHAIN_ID = "11155111"

# Blocks you want to fetch
#BLOCK_NUMBERS = list(range(11645660, 11645682))

BLOCK_NUMBERS=[11665136]

URL = "https://api.etherscan.io/v2/api"

OUTPUT_FILE = "../data/ehterblock.csv"


# --------------------------------------------------
# Create data directory
# --------------------------------------------------

os.makedirs("../data", exist_ok=True)


# --------------------------------------------------
# Read existing transaction hashes
# --------------------------------------------------

existing_tx_hashes = set()

if os.path.exists(OUTPUT_FILE):

    with open(OUTPUT_FILE, "r", newline="", encoding="utf-8") as file:

        reader = csv.DictReader(file)

        for row in reader:
            existing_tx_hashes.add(row["tx_hash"])


print(f"Existing transactions: {len(existing_tx_hashes)}")


# --------------------------------------------------
# Open CSV in append mode
# --------------------------------------------------

file_exists = os.path.exists(OUTPUT_FILE)

with open(
    OUTPUT_FILE,
    "a",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.writer(file)

    # Write header only if file doesn't exist
    if not file_exists:

        writer.writerow([
            "tx_hash",
            "from",
            "to",
            "value",
            "block_number"
        ])


    # --------------------------------------------------
    # Fetch each block
    # --------------------------------------------------

    for block_number in BLOCK_NUMBERS:

        print(f"\nFetching block {block_number}...")

        params = {
            "chainid": CHAIN_ID,
            "module": "proxy",
            "action": "eth_getBlockByNumber",
            "tag": hex(block_number),
            "boolean": "true",
            "apikey": API_KEY,
        }

        try:

            response = requests.get(
                URL,
                params=params,
                timeout=30
            )

            response.raise_for_status()

            data = response.json()

            # Check API response
            if "result" not in data or data["result"] is None:

                print(f"Failed to fetch block {block_number}")
                print(data)
                continue

            block = data["result"]

            transactions = block["transactions"]

            print(
                f"Block {block_number}: "
                f"{len(transactions)} transactions"
            )


            # --------------------------------------------------
            # Add transactions to CSV
            # --------------------------------------------------

            new_transactions = 0

            for tx in transactions:

                tx_hash = tx["hash"]

                # Don't add duplicate transaction
                if tx_hash in existing_tx_hashes:
                    continue

                writer.writerow([
                    tx_hash,
                    tx["from"],
                    tx["to"],
                    tx["value"],
                    block_number
                ])

                existing_tx_hashes.add(tx_hash)

                new_transactions += 1


            print(
                f"Added {new_transactions} new transactions"
            )


            # Small delay to avoid hitting API too aggressively
            time.sleep(0.2)


        except requests.RequestException as e:

            print(
                f"Network error while fetching "
                f"block {block_number}: {e}"
            )


print("\n--------------------------------")
print("Done!")
print(f"Total unique transactions: {len(existing_tx_hashes)}")
print(f"Saved to: {OUTPUT_FILE}")
print("--------------------------------")

Existing transactions: 0

Fetching block 11665136...
Block 11665136: 123 transactions
Added 123 new transactions

--------------------------------
Done!
Total unique transactions: 123
Saved to: ../data/ehterblock.csv
--------------------------------
